# Production Ranking Model — XGBoost `rank:ndcg`

Walks forward through quarterly cutoff dates, re-fitting a ranking model on
each expanding window, and evaluates the **top-of-the-list** metrics that map
to the screening product (top-10 / top-25 / top-quintile picks) instead of the
pooled regression metrics we already know are weak.

**Why `rank:ndcg` instead of regression:** the product asks "which names rank
highest", not "what will the return be". NDCG optimizes the head of the ranking
directly — in backtests it roughly doubled the top-10 excess return and
precision@10 vs. the regression baseline, at the cost of pooled rank IC.

**Purpose:** benchmark gate + production artifact. The last cell re-fits on all
history and saves the booster so the API/service layer can score any date cohort.


In [1]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from scipy.stats import spearmanr
from stockidence.storage import Warehouse

MODEL = Path.cwd()
if not (MODEL / "datasets").exists():
    MODEL = MODEL.parent  # run from Model/notebooks/ instead
if not (MODEL / "datasets").exists():
    raise SystemExit("run from Model/ or Model/notebooks/ (or fix PARQUET path)")
REPO = MODEL.parent                   # warehouse DB lives at repo/data/
DB = REPO / "data" / "stockidence.duckdb"
GRAIN = "quarterly"
PARQUET = MODEL / "datasets" / f"train_dataset_{GRAIN}.parquet"
ARTIFACT = MODEL / "artifacts"
ARTIFACT.mkdir(exist_ok=True)

EXCLUDED = {"NBIS", "BRK.B", "AMBP", "CRM", "BE", "FI"}
MODEL_NAME = "ranking_ndcg"

### 1. Load the curated training universe

`build_dataset.py` already writes only the training-universe tickers to the
parquet (see `--tickers` / `--tickers-file`); this cell applies the final
exclusion list and drops rows the feature set literally cannot see.

In [2]:
NEW_PRICE_FEATURES = [
    "price_to_sma200", "stddev_252", "max_drawdown_252", "atr_pct",
    "return_3m", "return_12m", "distance_from_52wk_high"]
CORE_STATIC_FUND = [
    "roe", "roa", "debt_equity", "current_ratio", "cash_to_assets", "fcf_to_assets"]
CATEGORICAL = ["sector"]
FEATURES = NEW_PRICE_FEATURES + CORE_STATIC_FUND + CATEGORICAL

df = pd.read_parquet(PARQUET)
TICKERS = sorted(set(df["ticker"].unique()) - EXCLUDED)
model_df = df[df["ticker"].isin(TICKERS)].dropna(subset=FEATURES + ["target_return"]).copy()
model_df["sector"] = model_df["sector"].astype("category")
print(f"universe: {len(TICKERS)} tickers | rows: {len(model_df)} | "
      f"{model_df['date'].min().date()} → {model_df['date'].max().date()}")

universe: 514 tickers | rows: 15874 | 2012-07-01 → 2026-04-01


### 2. Core-13 feature set

A/B on the pre-expansion parquet showed the 40-feature engineered set
(vol-scaled momentum, cross-sectional `rk_*` ranks, `rel_*` market-relative
momentum, 6m/24m returns) *halved* pooled rank IC (0.079 vs 0.156) and the
top-10 excess (+2.70 vs +5.85 pp/qtr) vs the plain 13 core metrics with
shallow trees — the derived columns are largely duplicative signal that
XGBoost overfits. Features here are the raw PIT parquet columns; no
engineer() step.


In [3]:
CORE13 = NEW_PRICE_FEATURES + CORE_STATIC_FUND

md = model_df.copy().reset_index(drop=True)
md["sector"] = md["sector"].astype("category")
print(f"CORE13: {len(CORE13)} features | model rows: {len(md)} | "
      f"tickers: {md['ticker'].nunique()}")

CORE13: 13 features | model rows: 15874 | tickers: 377


### 3. Walk-forward evaluation (expanding window, quarterly)

- Cutoff every quarter start from 2019 → 2025; train on `< cutoff`, test on
  `== cutoff` (one quarter of rows).
- Relevance grades = per-date quintile of realized forward return (0..4). Using
  grades on *train* rows only keeps the labels PIT and lets NDCG make full use
  of the order information.
- Group boundaries (`set_group`) tell XGBoost each date is a query group.

In [4]:
RANK_PARAMS = {
    "objective": "rank:ndcg", "tree_method": "hist",
    "max_depth": 3, "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8,
    "reg_alpha": 0.1, "reg_lambda": 1.0, "seed": 42, "nthread": -1,
}
N_GRADES = 5
CUTOFFS = pd.date_range("2019-01-01", "2025-04-01", freq="QS")

def grades_for(rows, n=N_GRADES):
    return rows.groupby("date")["target_return"].transform(
        lambda s: pd.qcut(s.rank(method="first"), n, labels=False)).astype(int)

R = []
for c in CUTOFFS:
    tr = md["date"] < c
    te = md["date"] == c
    if te.sum() < 5:
        continue
    y = grades_for(md.loc[tr])
    qid = md.loc[tr, "date"].astype("category").cat.codes.values
    dtr = xgb.DMatrix(md.loc[tr, CORE13], label=y)
    dtr.set_group(np.bincount(qid))
    bst = xgb.train(RANK_PARAMS, dtr, num_boost_round=150)
    p = bst.predict(xgb.DMatrix(md.loc[te, CORE13]))
    R.append(pd.DataFrame({"date": md.loc[te, "date"].values,
                           "ticker": md.loc[te, "ticker"].values,
                           "pred": p, "real": md.loc[te, "target_return"].values}))
R = pd.concat(R, ignore_index=True)
print(f"test rows: {len(R)}  quarters: {R['date'].nunique()}")

test rows: 7661  quarters: 26


### 4. Evaluation — head-of-ranking metrics

These are the numbers the screener is responsible for. `excess` is *market
relative*: each quarter, the top-K equal-weight mean minus that quarter's
equal-weight universe mean, so bull-market drift is removed. `precision@K` is
overlap between predicted top-K and realized top-K (random = K / n).

In [5]:
def head_metrics(R):
    rows = []
    for d, g in R.groupby("date"):
        g = g.sort_values("pred", ascending=False).reset_index(drop=True)
        uni = g["real"].mean()
        rr = g["real"].rank(ascending=False)
        rows.append({
            "date": d, "n": len(g), "uni": uni,
            "top10_exc": g.head(10)["real"].mean() - uni,
            "top25_exc": g.head(25)["real"].mean() - uni,
            "topQ_exc": g.head(len(g) // 5)["real"].mean() - uni,
            "prec10": rr.head(10).le(10).mean(),
            "prec25": rr.head(25).le(25).mean(),
        })
    return pd.DataFrame(rows)

def show(col):
    s = E[col]
    t = s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))
    print(f"  {col:10s} {s.mean()*100:+.2f} pp/qtr  t={t:+.2f}  "
          f"{100*(s > 0).mean():.0f}% of quarters")

ic = spearmanr(R["real"], R["pred"]).statistic
E = head_metrics(R)
print(f"rank IC (pooled): {ic:+.4f}   (bench random = 0; regression ~ +0.12)")
show("top10_exc"); show("top25_exc"); show("topQ_exc")
print(f"  precision@10: {E['prec10'].mean()*100:.1f}%   (random={10/E['n'].mean()*100:.1f}%)")
print(f"  precision@25: {E['prec25'].mean()*100:.1f}%   (random={25/E['n'].mean()*100:.1f}%)")
print()
R["year"] = R["date"].dt.year
for y, g in R.groupby("year"):
    sub = head_metrics(g)
    print(f"  {y}: top10 {sub['top10_exc'].mean()*100:+.2f} pp/qtr | "
          f"top25 {sub['top25_exc'].mean()*100:+.2f} | "
          f"topQ {sub['topQ_exc'].mean()*100:+.2f}  ({len(sub)} qtrs)")

rank IC (pooled): +0.1634   (bench random = 0; regression ~ +0.12)
  top10_exc  +3.90 pp/qtr  t=+1.44  73% of quarters
  top25_exc  +5.08 pp/qtr  t=+2.39  77% of quarters
  topQ_exc   +2.99 pp/qtr  t=+2.22  73% of quarters
  precision@10: 14.6%   (random=3.4%)
  precision@25: 22.8%   (random=8.5%)

  2019: top10 +6.96 pp/qtr | top25 +2.26 | topQ +1.22  (4 qtrs)
  2020: top10 +8.24 pp/qtr | top25 +11.92 | topQ +9.34  (4 qtrs)
  2021: top10 -3.50 pp/qtr | top25 -0.02 | topQ -0.21  (4 qtrs)
  2022: top10 -10.73 pp/qtr | top25 -0.20 | topQ +1.11  (4 qtrs)
  2023: top10 +15.04 pp/qtr | top25 +11.66 | topQ +4.84  (4 qtrs)
  2024: top10 +3.70 pp/qtr | top25 +1.93 | topQ -1.17  (4 qtrs)
  2025: top10 +11.25 pp/qtr | top25 +10.94 | topQ +8.56  (2 qtrs)


### 5. Benchmark: top-K vs the S&P 500

The universe already beats the S&P by design (equal-weight vs cap-weight). What
we care about is whether the *model's* top-K adds excess on top of both the
universe and the index. SPX forward quarter returns come from the FRED market
series, labeled as-of each test date.

In [6]:
def spx_forward(R):
    wh = Warehouse(DB)
    with wh.connect(read_only=True) as con:
        spx = pd.read_sql("SELECT date, spx FROM mart.m_fred_market "
                          "WHERE spx IS NOT NULL ORDER BY date", con)
    spx["date"] = pd.to_datetime(spx["date"])
    spx["q"] = spx["date"].dt.to_period("Q").dt.start_time.dt.normalize()
    qfwd = spx.groupby("q")["spx"].last().sort_index().pct_change().shift(-1)
    return R.assign(spx_fwd=R["date"].map(qfwd))

R2 = spx_forward(R).dropna(subset=["spx_fwd"])
rows = []
for d, g in R2.groupby("date"):
    g = g.sort_values("pred", ascending=False)
    rows.append({"date": d,
                 "top20": g.head(20)["real"].mean(),
                 "spx": g["spx_fwd"].iloc[0],
                 "uni": g["real"].mean()})
S = pd.DataFrame(rows)
def summ(col):
    v = S[col]
    t = v.mean() / (v.std(ddof=1) / np.sqrt(len(v)))
    return v.mean() * 100, t, 100 * (v > 0).mean()
m, t, h = summ("top20")
sx, st, sh = summ("spx")
ux, ut, uh = summ("uni")
print(f"quarters w/ SPX benchmark: {len(S)}")
print(f"  top-20 mean        : {m:+.2f}% / qtr")
print(f"  S&P 500 mean       : {sx:+.2f}% / qtr  (t={st:+.2f})")
print(f"  universe mean      : {ux:+.2f}% / qtr  (t={ut:+.2f})")
print(f"  top-20 vs S&P      : {m - sx:+0.2f} pp/qtr, beats S&P {h:.0f}% of quarters")
S["year"] = S["date"].dt.year
print()
for y, g in S.groupby("year"):
    v = g["top20"] - g["spx"]
    print(f"  {y}: top-20 vs S&P {v.mean()*100:+.2f} pp/qtr "
          f"({100*(v > 0).mean():.0f}%)")

quarters w/ SPX benchmark: 26
  top-20 mean        : +9.27% / qtr
  S&P 500 mean       : +3.74% / qtr  (t=+2.17)
  universe mean      : +4.04% / qtr  (t=+2.23)
  top-20 vs S&P      : +5.53 pp/qtr, beats S&P 73% of quarters

  2019: top-20 vs S&P +4.01 pp/qtr (75%)
  2020: top-20 vs S&P +15.56 pp/qtr (100%)
  2021: top-20 vs S&P -3.46 pp/qtr (50%)
  2022: top-20 vs S&P +0.81 pp/qtr (50%)
  2023: top-20 vs S&P +12.09 pp/qtr (100%)
  2024: top-20 vs S&P +2.54 pp/qtr (75%)
  2025: top-20 vs S&P +8.80 pp/qtr (100%)


/var/folders/jh/dwpc5mps01v9yszs4yfj4qn80000gn/T/ipykernel_46301/1371878757.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  spx = pd.read_sql("SELECT date, spx FROM mart.m_fred_market "


### 6. Production artifact

Re-fit on **all** history and save the booster + metadata so the service layer
can score requests without re-training. Grading uses every date's realized
returns — those are *outcomes*, not features, and the walk-forward above already
measured generalization; this fit is the deployed version.


In [7]:
yticks = grades_for(md)
qid = md["date"].astype("category").cat.codes.values
dm = xgb.DMatrix(md[CORE13], label=yticks)
dm.set_group(np.bincount(qid))
bst = xgb.train(RANK_PARAMS, dm, num_boost_round=150)

meta = {
    "model": MODEL_NAME,
    "objective": "rank:ndcg",
    "grain": GRAIN,
    "features": CORE13,
    "excluded": sorted(EXCLUDED),
    "n_rounds": 150,
    "params": RANK_PARAMS,
    "n_tickers": int(md["ticker"].nunique()),
    "n_rows": int(len(md)),
    "date_range": [str(md["date"].min().date()), str(md["date"].max().date())],
}
bst.save_model(str(ARTIFACT / f"{MODEL_NAME}.json"))
with open(ARTIFACT / f"{MODEL_NAME}.meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)
print(f"saved {ARTIFACT / MODEL_NAME}.json + .meta.json")
print(f"top features by gain: {', '.join(k for k,_ in sorted(bst.get_score(importance_type='gain').items(), key=lambda x:-x[1])[:10])}")


saved /Users/nativeongfuel/Stockidence/Model/artifacts/ranking_ndcg.json + .meta.json
top features by gain: max_drawdown_252, atr_pct, roa, return_12m, cash_to_assets, return_3m, distance_from_52wk_high, current_ratio, stddev_252, roe


### 7. Sanity: load artifact, score the latest quarter

Proves the artifact round-trips and gives a peek at what the live screener
would emit for the most recent cohort.


In [8]:
m2 = xgb.Booster()
m2.load_model(str(ARTIFACT / f"{MODEL_NAME}.json"))
last = md["date"] == md["date"].max()
Xl = xgb.DMatrix(md.loc[last, CORE13])
sc = m2.predict(Xl)
top = md.loc[last, ["ticker", "sector"]].assign(score=sc) \
            .sort_values("score", ascending=False).head(20)
top['rank'] = range(1, len(top)+1)
print(f"as of {md.loc[last, 'date'].iloc[0].date()} — top-20 cohort:")
print(top[["rank", "ticker", "sector", "score"]].to_string(index=False))

as of 2026-04-01 — top-20 cohort:
 rank ticker                 sector    score
    1   MRNA             Healthcare 1.005500
    2    CNC             Healthcare 0.782563
    3   DKNG Consumer Discretionary 0.719872
    4    NOW             Technology 0.677414
    5   SMCI             Technology 0.668999
    6    ACN             Technology 0.661105
    7   CSGP            Real Estate 0.637688
    8    TTD Communication Services 0.634216
    9   EPAM             Technology 0.596748
   10    ZTS             Healthcare 0.585013
   11     ZS             Technology 0.536621
   12   INTU             Technology 0.533659
   13   CTSH             Technology 0.529829
   14    TYL             Technology 0.524985
   15   SNOW             Technology 0.484522
   16   RDDT Communication Services 0.476200
   17    BSX             Healthcare 0.475957
   18   DECK Consumer Discretionary 0.469595
   19   WDAY             Technology 0.447463
   20    MOH             Healthcare 0.403779
